# Data overview and preprocessing

This notebook explores the raw `data/catalog.json` received from the company. We inspect its structure, product mix, feature-name frequency, and a reproducible sample of products and their feature descriptions. The examples use the catalog's default locale.

## 1. Load the catalog and inspect its structure

The catalog is nested: product fields live under `data`, localized text lives under `data.content[locale]`, and product features are stored as two parallel lists: `feature_titles` (the feature names) and `features` (the descriptions). The path check supports running this notebook either from this project folder or its parent workspace.

In [1]:
import html
import json
import random
from collections import Counter
from pathlib import Path
from pprint import pprint

catalog_path = Path("data/catalog.json")
if not catalog_path.is_file():
    catalog_path = Path("CTFDS-Project/data/catalog.json")
if not catalog_path.is_file():
    raise FileNotFoundError("Could not find data/catalog.json from the notebook or workspace folder.")

catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
products = catalog.get("products")
if not isinstance(products, list):
    raise ValueError("Expected catalog.json to contain a 'products' list.")

default_locale = catalog.get("default_locale", "en-GB")
print(f"Catalog: {catalog_path.resolve()}")
print(f"Schema version: {catalog.get('schema_version')}")
print(f"Products: {len(products)}")
print(f"Default locale: {default_locale}")
print(f"Locales: {catalog.get('locales', [])}")
print("Top-level keys:", list(catalog.keys()))
print("Keys in a product record:", list(products[0].keys()) if products else [])

Catalog: C:\Users\greta\Desktop\2_semester\comp tolls fr data science\CTFDS-Project\data\catalog.json
Schema version: 1.0.0
Products: 208
Default locale: en-GB
Locales: ['en-GB', 'de-DE', 'es-ES', 'fr-FR', 'it-IT', 'nl-NL', 'sv-SE', 'da-DK', 'nb-NO']
Top-level keys: ['schema_version', 'default_locale', 'locales', 'currencies', 'inheritance', 'products']
Keys in a product record: ['id', 'data', 'media', 'variants']


## 2. Product and feature statistics

Feature-name counts below describe each parent product's default-locale content and are exact after trimming whitespace and decoding HTML entities; capitalization and wording variants remain separate. We also summarize variants and check whether each product has the same number of feature names and descriptions, since the raw file appears to associate these by list position.

In [2]:
product_types = Counter()
genders = Counter()
locale_coverage = Counter()
feature_name_counts = Counter()
variant_total = 0
products_with_variants = 0
variants_with_default_locale_content_override = 0
product_summaries = []
length_mismatches = []
blank_feature_descriptions = []

for product in products:
    variants = product.get("variants") or []
    variant_total += len(variants)
    products_with_variants += bool(variants)
    for variant in variants:
        override_content = ((variant.get("overrides") or {}).get("content") or {})
        variants_with_default_locale_content_override += default_locale in override_content

    product_data = product.get("data") or {}
    classification = product_data.get("classification") or {}
    content_by_locale = product_data.get("content") or {}
    locale_coverage.update(content_by_locale.keys())
    content = content_by_locale.get(default_locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []

    product_type = classification.get("product_type", "[missing]")
    gender = classification.get("gender", "[missing]")
    product_types[product_type] += 1
    genders[gender] += 1

    clean_feature_names = [html.unescape(str(feature_name)).strip() for feature_name in feature_names]
    clean_feature_names = [feature_name for feature_name in clean_feature_names if feature_name]
    feature_name_counts.update(clean_feature_names)
    product_summaries.append({
        "id": product.get("id", "[missing]"),
        "name": content.get("name", "[missing name]"),
        "product_type": product_type,
        "gender": gender,
        "feature_count": len(feature_names),
        "description_count": len(descriptions),
    })

    if len(feature_names) != len(descriptions):
        length_mismatches.append({
            "id": product.get("id", "[missing]"),
            "name": content.get("name", "[missing name]"),
            "feature_names": len(feature_names),
            "descriptions": len(descriptions),
        })
    for index, feature_name in enumerate(feature_names):
        if index >= len(descriptions) or not str(descriptions[index]).strip():
            blank_feature_descriptions.append((product.get("id"), feature_name))

print(f"Parent products: {len(products)}")
print(f"Variants: {variant_total} across {products_with_variants} products")
print(f"Variants with a {default_locale} content override: {variants_with_default_locale_content_override}")
print(f"Localized product-content records: {dict(sorted(locale_coverage.items()))}")
print(f"Feature-name occurrences: {sum(feature_name_counts.values())}")
print(f"Distinct exact feature names: {len(feature_name_counts)}")
print(f"Products with different feature-name/description counts: {len(length_mismatches)}")
print(f"Feature names without a non-empty description at the same position: {len(blank_feature_descriptions)}")

print("\nProducts by type:")
for name, count in product_types.most_common():
    print(f"  {name}: {count}")

print("\nProducts by gender:")
for name, count in genders.most_common():
    print(f"  {name}: {count}")

print("\nMost frequent feature names:")
for feature_name, count in feature_name_counts.most_common(15):
    print(f"  {count:>3}  {feature_name}")

Parent products: 208
Variants: 444 across 208 products
Variants with a en-GB content override: 72
Localized product-content records: {'da-DK': 208, 'de-DE': 208, 'en-GB': 208, 'es-ES': 208, 'fr-FR': 208, 'it-IT': 208, 'nb-NO': 208, 'nl-NL': 208, 'sv-SE': 208}
Feature-title occurrences: 1073
Distinct exact feature titles: 444
Products with different title/description counts: 7
Titles without a non-empty description at the same position: 13

Products by type:
  Cycling Wear Tops: 42
  Cycling Wear Bottoms: 31
  Socks: 25
  Headwear: 24
  Shoe Covers: 21
  Base Layers: 21
  Gloves Cycling Winter: 15
  Warmers: 12
  Gloves Cycling Summer Short: 7
  Gloves Cycling Summer Long: 6
  Accessories: 2
  Cycling Wear One-Pieces: 2

Products by gender:
  Unisex: 115
  Men's: 54
  Women's: 38
  Junior: 1

Most frequent feature titles:
   53  Reflective details
   23  Highly breathable
   15  Durable water repellent (DWR) treatment
   15  Stretch arch support
   14  Anti-odour treatment
   12  Reflec

## 3. Inspect sample products and features

The sample is reproducible and includes a product with mismatched feature-list lengths when one exists. If the two lists have different lengths, the code prints them separately rather than using `zip`, which would silently drop unmatched entries and imply pairings we cannot verify.

In [3]:
sample_products = random.Random(42).sample(products, k=min(3, len(products)))
mismatch_ids = {row["id"] for row in length_mismatches}
mismatch_example = next((product for product in products if product.get("id") in mismatch_ids), None)
if mismatch_example is not None and all(p.get("id") != mismatch_example.get("id") for p in sample_products):
    sample_products.append(mismatch_example)

for product in sample_products:
    product_data = product.get("data") or {}
    classification = product_data.get("classification") or {}
    content = (product_data.get("content") or {}).get(default_locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []

    print(f"\nProduct {product.get('id')} — {content.get('name', '[missing name]')}")
    print(f"Type: {classification.get('product_type', '[missing]')} | Gender: {classification.get('gender', '[missing]')}")
    variants = product.get("variants") or []
    print(f"Variants: {len(variants)}")
    for variant in variants[:2]:
        color = (variant.get("color") or {}).get("name", "[color not specified]")
        print(f"  Variant {variant.get('id', '[missing id]')} | color: {color}")
    if len(feature_names) == len(descriptions):
        for feature_name, description in zip(feature_names, descriptions):
            print(f"  - {feature_name}: {description or '[empty description]'}")
    else:
        print(f"  Feature names ({len(feature_names)}):")
        for feature_name in feature_names:
            print(f"    - {feature_name}")
        print(f"  Descriptions ({len(descriptions)}; pairing not assumed):")
        for description in descriptions:
            print(f"    - {description or '[empty description]'}")


Product 6111 — GripGrab RIDE Cycling Shorts
Type: Cycling Wear Bottoms | Gender: Men's
Variants: 1
  Variant 611101 | color: Black
  - High-quality construction: Crafted from high-quality fabrics offering a balanced level of stretch and support for enhanced comfort
  - Quality padding (LIEGE HP): Integrates a high-quality cycling pad from Elastic Interface (LIEGE HP) to deliver outstanding comfort during long rides
  - 1 top entry leg pocket: Features a convenient stretchy pocket, lying flat against the leg for easy access to your phone or snacks throughout your ride
  - Wide elastic hems: Ensure a secure fit on the legs with wide elastic hems and silicone grippers
  - Wicking waistband: Stays comfortably in place during your rides thanks to the stretch waistband lined with moisture-wicking mesh

Product 1075 — GripGrab Primavera 2 Merino Spring-Autumn Gloves
Type: Gloves Cycling Winter | Gender: Unisex
Variants: 1
  Variant 107501 | color: Black
  - Luxurious soft merino blend: A mer

In [4]:
print("Examples for the three most frequent feature names:")
for feature_name, _ in feature_name_counts.most_common(3):
    print(f"\n{feature_name}")
    shown = 0
    for product in products:
        product_data = product.get("data") or {}
        content = (product_data.get("content") or {}).get(default_locale) or {}
        feature_names = content.get("feature_titles") or []
        descriptions = content.get("features") or []
        if len(feature_names) != len(descriptions):
            continue
        for raw_feature_name, description in zip(feature_names, descriptions):
            if html.unescape(str(raw_feature_name)).strip() == feature_name:
                print(f"  {content.get('name', '[missing name]')}: {description or '[empty description]'}")
                shown += 1
                if shown == 3:
                    break
        if shown == 3:
            break

Examples for the three most frequent feature titles:

Reflective details
  GripGrab Waterproof Helmet Cover: for visibility in the traffic
  GripGrab W's PACR Thermal Bib Shorts: For visibility in traffic
  GripGrab PACR Windproof Lightweight Vest: Enhance your visibility to other road users in low light conditions

Highly breathable
  GripGrab FastStream Aero Cycling Socks: The foot section of the sock, crafted from Coolmax® yarn, offers great breathability
  GripGrab RIDE Winter Long Sleeve Base Layer: providing excellent ventilation
  GripGrab Ultralight Mesh Sleeveless Base Layer 3-Pack: Ultralight open mesh fabric allows heat and moisture to escape from within, keeping you cool in hot summer conditions

Durable water repellent (DWR) treatment
  GripGrab W's PACR Thermal Bib Shorts: Providing effective water repellency
  GripGrab RIDE 2 Waterproof Winter Gloves: providing effective water repellency
  GripGrab RIDE 2 Windproof Spring-Autumn Gloves: providing effective water repellen

## 4. Why preprocessing is needed and what it will do

The raw catalog is useful for product-level inspection, but it is not yet a clean feature table. Content is nested by product and locale; each product may also have variant-level overrides. Feature names and descriptions are stored in separate positional lists, and some products have different list lengths or empty descriptions. Repeated feature names also occur across products with different wording, capitalization, or HTML entities.

We will turn the catalog into two smaller files for the analysis that follows:

- `data/features_en.json`: one record per cleaned feature name, with its associated descriptions; descriptions that cannot safely be paired are kept separately.
- `data/products_en.json`: one record per product, with its product type and feature names, for later evaluation.

The JSON files keep the catalog's field names (`attribute_records`, `title`, `titles`); in the text we call these feature names.

The code selects the default English locale, includes product-level and variant-override content, decodes HTML entities, trims text, and aggregates repeated feature names. When a product's feature-name and description lists have different lengths, it does not guess the pairings: it keeps those descriptions as unpaired instead of silently losing them with `zip`.

## 5. Preprocess the catalog

Run this after the overview cells. The outputs are written next to `catalog.json` and are used by the following analysis and clustering notebooks.

In [5]:
import html
import json

data_dir = catalog_path.parent
features_path = data_dir / "features_en.json"
products_path = data_dir / "products_en.json"

blocks = []
for raw_product in products:
    product_type = raw_product["data"]["classification"]["product_type"]
    product_id = raw_product["id"]
    content = (raw_product.get("data") or {}).get("content") or {}
    blocks.append((product_id, product_type, content.get(default_locale, {})))

    for variant in raw_product.get("variants") or []:
        variant_content = ((variant.get("overrides") or {}).get("content") or {})
        blocks.append((product_id, product_type, variant_content.get(default_locale, {})))

feature_name_records = {}
product_records = {}
unpaired_descriptions = []

for product_id, product_type, content in blocks:
    feature_names = [html.unescape(feature_name).strip() for feature_name in content.get("feature_titles") or []]
    descriptions = [html.unescape(description).strip() for description in content.get("features") or []]

    product = product_records.setdefault(
        product_id,
        {"product_id": product_id, "product_type": product_type, "titles": set()},
    )
    product["titles"].update(feature_names)

    for feature_name in feature_names:
        feature_name_records.setdefault(feature_name, {"title": feature_name, "descriptions": []})

    if len(feature_names) != len(descriptions):
        unpaired_descriptions.extend(descriptions)
        continue

    for feature_name, description in zip(feature_names, descriptions):
        feature_name_records[feature_name]["descriptions"].append(description)

for product in product_records.values():
    product["titles"] = sorted(product["titles"])

features_path.write_text(
    json.dumps(
        {
            "attribute_records": list(feature_name_records.values()),
            "unpaired_descriptions": unpaired_descriptions,
        },
        ensure_ascii=False,
        indent=1,
    ),
    encoding="utf-8",
)
products_path.write_text(
    json.dumps(list(product_records.values()), ensure_ascii=False, indent=1),
    encoding="utf-8",
)

print(f"Wrote {len(feature_name_records)} unique feature names to {features_path}")
print(f"Kept {len(unpaired_descriptions)} descriptions that could not be paired safely")
print(f"Wrote {len(product_records)} products to {products_path}")

Wrote 445 unique feature titles to data\features_en.json
Kept 22 descriptions that could not be paired safely
Wrote 208 products to data\products_en.json
